# Reto de Laboratorio: Preparando la Matriz Tensorial de la Liga ⚽🤖


Contexto de Negocio:
El equipo de analítica quedó fascinado con el Almacén de Datos (OLAP) que construyeron la semana pasada. Ahora, el departamento de Inteligencia Artificial quiere dar el siguiente paso: entrenar una Red Neuronal para predecir cuántos goles anotará un jugador la próxima temporada.El Científico de Datos les acaba de enviar la tabla resultante del cubo OLAP, pero se detuvo por un error de Scikit-Learn: el algoritmo no sabe leer posiciones como "Forward" ni jerarquías como "Oro".Su misión como Ingenieros de Machine Learning es aplicar Feature Engineering para traducir esta tabla humana a una Matriz $\mathbf{X}$ puramente matemática, sin destruir la lógica vectorial.

In [44]:
import pandas as pd
import numpy as np

# 1. Ingesta de datos desde el repositorio crudo
url = "https://raw.githubusercontent.com/mricardo89/data-mining/main/Unit-III/datasets/players_data-2025_2026.csv"
df_scouting = pd.read_csv(url)

# 2. SIMULACIÓN: El equipo de analistas agregó una columna de jerarquía de talento
# (Evaluación cualitativa del jugador)
np.random.seed(42)
df_scouting['scout_tier'] = np.random.choice(['Bronce', 'Plata', 'Oro', 'Diamante'], size=len(df_scouting))

# Filtramos las columnas que usaremos para el modelo predictivo
columnas_ml =  ['Player', 'Squad', 'Age', 'Pos', 'scout_tier', 'Min', 'Gls']
matriz_base = df_scouting[columnas_ml].copy()

print("--- DATOS BASE PARA EL MODELO (Legibles para humanos) ---")
display(matriz_base.head())

--- DATOS BASE PARA EL MODELO (Legibles para humanos) ---


,Player,Squad,Age,Pos,scout_tier,Min,Gls
0,Brenden Aaronson,Leeds United,24.0,"MF,FW",Oro,2449,4
1,Jerome Abbey,Wolves,15.0,MF,Diamante,17,0
2,Zach Abbott,Nottingham Forest,19.0,DF,Bronce,164,0
3,Jones El-Abdellaoui,Celta Vigo,19.0,"MF,FW",Oro,681,2
4,Himad Abdelli,Marseille,25.0,MF,Oro,138,0


### FASE 1: El Traductor Jerárquico (Label Encoding)

Observa la columna `scout_tier`. Contiene valores de texto: 'Bronce', 'Plata', 'Oro', 'Diamante'. Esta variable tiene un **orden natural** claro (Diamante es superior a Bronce).

**Tu tarea:**
Crea un diccionario de mapeo que asigne números enteros respetando la jerarquía natural del talento (iniciando en 1). Luego, aplica este mapeo a la base de datos para crear una nueva columna llamada `scout_encoded`.

In [45]:
mapeo_scout = {
    'Bronce': 1,
    'Plata': 2,
    'Oro': 3,
    'Diamante': 4
}
matriz_base['scout_encoded'] = matriz_base['scout_tier'].map(mapeo_scout)

print("Resultado del Label Encoding:")
display(matriz_base[['Player', 'scout_tier', 'scout_encoded']].head())

Resultado del Label Encoding:


,Player,scout_tier,scout_encoded
0,Brenden Aaronson,Oro,3
1,Jerome Abbey,Diamante,4
2,Zach Abbott,Bronce,1
3,Jones El-Abdellaoui,Oro,3
4,Himad Abdelli,Oro,3


### FASE 2: El Traductor Nominal (One-Hot Encoding)

Ahora observa la columna `position`. Contiene valores como 'Forward' (Delantero), 'Midfielder' (Mediocampista), 'Defender' (Defensa), etc.
Esta variable es **Nominal**. Si le asignamos números (1 al Delantero y 3 al Defensa), la Red Neuronal creerá falsamente que un Defensa "vale el triple" que un Delantero.

**Tu tarea:**
Utiliza la función de Pandas correspondiente para destruir la columna `position` y expandirla en múltiples columnas booleanas (0 y 1). Guarda el resultado en una nueva variable llamada `matriz_X`.

In [46]:
# CÓDIGO ESQUELETO PARA EL ALUMNO:
matriz_X = matriz_base.copy()
matriz_X['Forward'] = matriz_X['Pos'].str.contains('FW', na=False).astype(int)
matriz_X['Midfielder'] = matriz_X['Pos'].str.contains('MF', na=False).astype(int)
matriz_X['Defender'] = matriz_X['Pos'].str.contains('DF', na=False).astype(int)


print("Resultado del One-Hot Encoding:")
display(matriz_X.head())

Resultado del One-Hot Encoding:


,Player,Squad,Age,Pos,scout_tier,Min,Gls,scout_encoded,Forward,Midfielder,Defender
0,Brenden Aaronson,Leeds United,24.0,"MF,FW",Oro,2449,4,3,1,1,0
1,Jerome Abbey,Wolves,15.0,MF,Diamante,17,0,4,0,1,0
2,Zach Abbott,Nottingham Forest,19.0,DF,Bronce,164,0,1,0,0,1
3,Jones El-Abdellaoui,Celta Vigo,19.0,"MF,FW",Oro,681,2,3,1,1,0
4,Himad Abdelli,Marseille,25.0,MF,Oro,138,0,3,0,1,0


### FASE 3: Purgando la Basura Humana

La Inteligencia Artificial está casi lista para devorar los datos. Sin embargo, si le enviamos la matriz ahora mismo, el algoritmo fallará porque todavía hay texto puro estorbando.

**Tu tarea:**
Elimina de `matriz_X` todas las columnas que ya no le sirven al algoritmo:
1. Las columnas de texto original que ya transformaste (como `scout_tier`).
2. Las columnas descriptivas/operativas que no aportan valor matemático universal (`player_name`, `team_name`).

In [47]:
matriz_X.drop(columns=['scout_tier'], inplace=True)
matriz_X.drop(columns=['Player', 'Squad'], inplace=True)
matriz_X.drop(columns=['Pos'], inplace=True)

print("--- MATRIZ X DEFINITIVA (Lista para Scikit-Learn / PyTorch) ---")
display(matriz_X.head())
print("\nTipos de datos finales (Todos deben ser numéricos):")
print(matriz_X.dtypes)

--- MATRIZ X DEFINITIVA (Lista para Scikit-Learn / PyTorch) ---


,Age,Min,Gls,scout_encoded,Forward,Midfielder,Defender
0,24.0,2449,4,3,1,1,0
1,15.0,17,0,4,0,1,0
2,19.0,164,0,1,0,0,1
3,19.0,681,2,3,1,1,0
4,25.0,138,0,3,0,1,0



Tipos de datos finales (Todos deben ser numéricos):
Age              float64
Min                int64
Gls                int64
scout_encoded      int64
Forward            int64
Midfielder         int64
Defender           int64
dtype: object


### 🧠 FASE 4: Defensa Arquitectónica (Responde aquí mismo)

Imagina que el CTO de la empresa revisa tu código y te hace dos preguntas críticas. Responde con argumentos de ingeniería:

**Pregunta 1:** *"Noté que borraste la columna `team_name` en lugar de aplicarle One-Hot Encoding. En nuestra base de datos mundial de la FIFA tenemos 4,000 equipos diferentes. Si le hubieras aplicado One-Hot Encoding a la columna de equipos, ¿qué le habría pasado a la estructura matemática de tu Matriz X y a la memoria RAM de nuestro servidor?"*
> **Tu Respuesta:** Tomemos como ejemplo el caso de las posiciones de los jugadores: los registros eran texto y se utilizó la tecnica de one hot encoding para poder manejar estos datos, como sólo existen 3 categorias diferentes para posición, se agregaron únicamente 3 columnas, pero si quisiéramos utilizar el mismo método para el equipo, se agregarían 4,000 nuevas columnas al dataframe lo cual representa una cantidad descontrolada de registros en 0 y resultaría en un uso disparado de la memoria RAM que podría causar que se quede sin ella (memoria).

**Pregunta 2:** *"¿Por qué usamos Label Encoding (1, 2, 3, 4) para la evaluación del scout (`scout_tier`), pero no usamos esa misma técnica matemática para la posición en el campo (`position`)?"*
> **Tu Respuesta:** Porque en scout tier hablamos de qué tan buenos son los jugadores, al modificar los registros para que sean numéricos estamos indicando que éstos poseen valor según su número asignado porque es una variable ordinal (5>1, por lo que 5 es mejor que 1). En el caso de la posicion del jugador, no hablamos de una caracteristica que le otorgue mayor valor, solo es una caracteristica más (variable nominal), por lo que no quisieramos que la computadora lo interprete como un factor para rankear de mayor a menor. 